# Notebook 06 — Debugging with Event History

This notebook demonstrates how to debug Temporal Workflows using event history, the Web UI, and CLI tools.

## What You Will Learn
- How to read event history
- Common failure patterns and how to identify them
- Using the CLI and Web UI for debugging
- Debugging workflow: identify → reproduce → fix → verify

## Event Types Reference

| Event | Meaning |
|-------|---------|
| `WorkflowExecutionStarted` | Workflow received input and began |
| `WorkflowTaskScheduled` | Task placed on queue |
| `WorkflowTaskStarted` | Worker picked up the task |
| `WorkflowTaskCompleted` | Worker finished processing |
| `ActivityTaskScheduled` | Activity dispatched |
| `ActivityTaskStarted` | Activity began executing |
| `ActivityTaskCompleted` | Activity returned a result |
| `ActivityTaskFailed` | Activity raised an exception |
| `ActivityTaskTimedOut` | Activity exceeded timeout |
| `TimerStarted` | Durable timer created |
| `TimerFired` | Timer elapsed |
| `WorkflowExecutionCompleted` | Workflow finished successfully |
| `WorkflowExecutionFailed` | Workflow terminated with error |

In [ ]:
# Simulate a successful event history
successful_history = [
    "WorkflowExecutionStarted",
    "WorkflowTaskScheduled",
    "WorkflowTaskStarted",
    "WorkflowTaskCompleted",
    "ActivityTaskScheduled     [get_distance]",
    "ActivityTaskStarted",
    "ActivityTaskCompleted     [result: {km: 20}]",
    "TimerStarted             [3s]",
    "TimerFired",
    "ActivityTaskScheduled     [send_bill]",
    "ActivityTaskStarted",
    "ActivityTaskCompleted     [result: SUCCESS]",
    "WorkflowExecutionCompleted",
]

print("✓ Successful Workflow Event History:")
print("=" * 55)
for i, event in enumerate(successful_history, 1):
    print(f"  {i:>2}. {event}")

In [ ]:
# Simulate a failed event history
failed_history = [
    "WorkflowExecutionStarted",
    "WorkflowTaskScheduled",
    "WorkflowTaskStarted",
    "WorkflowTaskCompleted",
    "ActivityTaskScheduled     [get_distance]",
    "ActivityTaskStarted",
    "ActivityTaskCompleted     [result: {km: 30}]",
    "WorkflowExecutionFailed   [customer lives outside the service area]",
]

print("✗ Failed Workflow Event History:")
print("=" * 60)
for i, event in enumerate(failed_history, 1):
    marker = "  " if "Failed" not in event else "⚠️"
    print(f"  {marker} {i:>2}. {event}")

print("\nDiagnosis: get_distance returned 30 km, which exceeds the 25 km limit.")
print("The Workflow raised ApplicationError because the customer is outside delivery area.")

In [ ]:
# Simulate an activity retry sequence
retry_history = [
    "ActivityTaskScheduled     [send_bill]",
    "ActivityTaskStarted",
    "ActivityTaskFailed        [error: connection timeout]",
    "ActivityTaskScheduled     [send_bill — retry #1]",
    "ActivityTaskStarted",
    "ActivityTaskFailed        [error: connection timeout]",
    "ActivityTaskScheduled     [send_bill — retry #2]",
    "ActivityTaskStarted",
    "ActivityTaskCompleted     [result: SUCCESS]",
]

print("Activity Retry Sequence:")
print("=" * 55)
for i, event in enumerate(retry_history, 1):
    marker = "  "
    if "Failed" in event:
        marker = "⚠️"
    elif "Completed" in event:
        marker = "✓ "
    print(f"  {marker} {i:>2}. {event}")

print("\nThe Activity failed twice due to connection timeouts, then succeeded on retry #2.")

## CLI Debugging Commands

```bash
# List all workflows
temporal workflow list

# Show event history
temporal workflow show --workflow-id pizza-workflow-order-XD001

# Describe a workflow (status, start time, etc.)
temporal workflow describe --workflow-id pizza-workflow-order-XD001

# Show history as JSON (for detailed inspection)
temporal workflow show --workflow-id pizza-workflow-order-XD001 --output json
```

## Web UI Debugging Steps

1. Open `http://localhost:8233`
2. Find the Workflow by ID or filter by status
3. Click the Workflow to see details
4. Examine the **Event History** tab
5. Look for `ActivityTaskFailed` or `WorkflowExecutionFailed` events
6. Expand the event to see the error message and stack trace

## Debugging Checklist

```
1. [ ] Check Workflow status in the Web UI
2. [ ] Examine event history for failed events
3. [ ] Read the error message and stack trace
4. [ ] Check Activity inputs — were they correct?
5. [ ] Check Activity code — is the logic right?
6. [ ] Write a test to reproduce the bug
7. [ ] Fix the bug
8. [ ] Verify with the test
9. [ ] Re-run the Workflow to confirm
```

In [ ]:
# Debugging exercise: identify the bug
def buggy_send_bill(amount: int) -> int:
    """Simulates a buggy billing function."""
    charge_amount = amount
    
    # Bug: discount is applied even when amount <= 3000
    # Should be: if charge_amount > 3000
    if charge_amount > 0:  # BUG!
        charge_amount -= 500
    
    if charge_amount < 0:
        raise ValueError(f"invalid charge amount: {charge_amount}")
    
    return charge_amount


# Test cases
test_cases = [
    (4000, "Expected: 3500 (discount applied)"),
    (2600, "Expected: 2600 (no discount)"),
    (300, "Expected: 300 (no discount)"),
]

print("Bug Demonstration:")
print("-" * 45)
for amount, description in test_cases:
    try:
        result = buggy_send_bill(amount)
        print(f"  Amount: {amount:>5} → Charged: {result:>5}  {description}")
    except ValueError as e:
        print(f"  Amount: {amount:>5} → ERROR: {e}")

print("\n⚠️ Bug: discount is applied to ALL positive amounts, not just > 3000")

In [ ]:
# Fixed version
def fixed_send_bill(amount: int) -> int:
    """Fixed billing function — discount only for amounts > 3000."""
    charge_amount = amount
    
    if charge_amount > 3000:  # FIXED
        charge_amount -= 500
    
    if charge_amount < 0:
        raise ValueError(f"invalid charge amount: {charge_amount}")
    
    return charge_amount


print("Fixed Version:")
print("-" * 45)
for amount, description in test_cases:
    try:
        result = fixed_send_bill(amount)
        print(f"  Amount: {amount:>5} → Charged: {result:>5}  {description}")
    except ValueError as e:
        print(f"  Amount: {amount:>5} → ERROR: {e}")

print("\n✓ Fixed: discount correctly applied only to amounts > 3000")

## Summary

- Use the **Web UI** and **CLI** to inspect event history
- Look for `ActivityTaskFailed` and `WorkflowExecutionFailed` events
- Follow the debugging checklist: identify → reproduce → fix → verify
- Write **tests** to reproduce bugs before fixing them
- Use **mocked Activities** to isolate the component under test

This completes the notebook series. Happy debugging!